# ViT-Tiny CIFAR-100 — Colab 训练

使用前先把 `cifar100_code.zip` 上传到 Google Drive 根目录（`drive.google.com` 拖拽上传）。

## 1. 安装依赖

In [ ]:
!pip install pyyaml tensorboard -q

## 2. 挂载 Drive + 解压代码

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

drive_zip = '/content/drive/MyDrive/cifar100_code.zip'
project_dir = '/content/cifar100'

if os.path.exists(drive_zip):
    os.makedirs(project_dir, exist_ok=True)
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(project_dir)
    print(f'已解压到 {project_dir}')
else:
    raise FileNotFoundError(
        f'未找到 {drive_zip}\n'
        '请先将 cifar100_code.zip 上传到 Google Drive 根目录'
    )

print('\n项目文件:')
for f in sorted(os.listdir(project_dir)):
    print(f'  {f}')

## 3. CIFAR-100 数据集（自动缓存到 Drive）

In [ ]:
import torchvision, os, shutil

dataset_root = '/content/cifar100_data'
drive_dataset = '/content/drive/MyDrive/cifar100_data'

# 优先从 Drive 恢复（几秒完成）
if os.path.exists(os.path.join(drive_dataset, 'cifar-100-python')):
    if not os.path.exists(os.path.join(dataset_root, 'cifar-100-python')):
        print('从 Drive 恢复数据集...')
        shutil.copytree(drive_dataset, dataset_root, dirs_exist_ok=True)
    print(f'数据集已就绪 → {dataset_root}')
else:
    # 首次下载，然后备份到 Drive
    print('首次下载 CIFAR-100...')
    os.makedirs(dataset_root, exist_ok=True)
    torchvision.datasets.CIFAR100(root=dataset_root, train=True, download=True)
    torchvision.datasets.CIFAR100(root=dataset_root, train=False, download=True)
    print('备份到 Drive...')
    os.makedirs(drive_dataset, exist_ok=True)
    shutil.copytree(dataset_root, drive_dataset, dirs_exist_ok=True)
    print(f'下载完成，已备份到 Drive → {drive_dataset}')

## 4. 检查 GPU

In [16]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'显存: {mem:.1f} GB')
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('警告: 未检测到 GPU，训练会非常慢')

CUDA: True
GPU: Tesla T4
显存: 14.6 GB
PyTorch: 2.11.0+cu128
CUDA: 12.8


## 5. 调参（改这里就行，不用重新上传 zip）

In [17]:
import yaml, os

config_path = '/content/cifar100/config_colab_vit.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# ======== 在这里改参数 ========
cfg['data']['train_batch_size'] = 1024     # 训练 batch size
cfg['data']['test_batch_size'] = 2048      # 测试 batch size
cfg['train']['lr'] = 0.0005                # 学习率
cfg['train']['epoch_num'] = 300            # 总轮次
cfg['train']['warmup_epochs'] = 10         # warmup 轮次
cfg['train']['label_smoothing'] = 0.1      # 标签平滑
cfg['train']['weight_decay'] = 0.05        # weight decay
cfg['train']['mixup_alpha'] = 0.2          # Mixup alpha
cfg['train']['cutmix_prob'] = 0.5          # CutMix 概率
# ==============================

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('配置已更新:')
print(f'  batch_size: {cfg["data"]["train_batch_size"]}')
print(f'  lr: {cfg["train"]["lr"]}')
print(f'  epochs: {cfg["train"]["epoch_num"]}')
print(f'  warmup: {cfg["train"]["warmup_epochs"]}')

配置已更新:
  batch_size: 1024
  lr: 0.0005
  epochs: 300
  warmup: 10


## 6. 启动 ViT 训练

In [ ]:
import os
os.chdir('/content/cifar100')
print(f'工作目录: {os.getcwd()}')
print()
!python train.py --config config_colab_vit.yaml --drive-sync /content/drive/MyDrive/cifar100_runs/vit

工作目录: /content/cifar100

GPU: Tesla T4
CUDA 版本: 12.8
PyTorch 版本: 2.11.0+cu128
GPU 显存: 14.6 GB
使用设备: cuda:0
当前模型: vit
模型参数: {'attention_dropout': 0.0, 'depth': 6, 'drop_path_rate': 0.1, 'dropout': 0.1, 'embed_dim': 256, 'image_size': 32, 'in_channels': 3, 'mlp_ratio': 4.0, 'num_classes': 100, 'num_heads': 8, 'patch_size': 4}
AMP 已启用 (torch.amp.GradScaler)
模型参数量: 4,794,212 (4.79M)
可训练参数: 4,794,212 (4.79M)
EMA 已启用, decay=0.999
训练集数量: 50000
测试集数量: 10000
优化器: adamw
模型保存目录: /content/cifar100/models/vit
日志保存目录: /content/cifar100/logs/vit
Mixup alpha=0.2, CutMix alpha=1.0, prob=0.5
Epoch [1/300] | Train Loss: 4.5378 | Train Acc: 0.0257 | Epoch: 00:00:43 | ETA: 03:37:04 | GPU Mem: 5571/6004 MB
         | 新最佳模型已保存: /content/cifar100/models/vit/cifar100_best.pth (Acc=0.0141)
         | 已同步 best 到 Drive: /content/drive/MyDrive/cifar100_runs/vit/cifar100_best.pth
         | Test Loss: 4.6286 | Test Acc: 0.0141 | Best: 0.0141 (Epoch 1) | EarlyStop: 0/50

Epoch [2/300] | Train Loss: 4.4393 | Train Ac

## 7. 查看训练结果

In [21]:
import json, os

summary_path = '/content/cifar100/results/vit_summary.json'
if os.path.exists(summary_path):
    with open(summary_path) as f:
        s = json.load(f)
    print('训练摘要:')
    for k, v in s.items():
        print(f'  {k}: {v}')
else:
    print('尚未生成训练摘要')

drive_dir = '/content/drive/MyDrive/cifar100_runs/vit'
if os.path.isdir(drive_dir):
    files = sorted(os.listdir(drive_dir))
    print(f'\nDrive 中的文件 ({len(files)} 个):')
    for f in files:
        size = os.path.getsize(os.path.join(drive_dir, f)) / 1024**2
        print(f'  {f}  ({size:.1f} MB)')

训练摘要:
  model: vit
  total_params: 4794212
  trainable_params: 4794212
  best_test_accuracy: 0.6824
  best_epoch: 219
  total_epochs: 42
  optimizer: adamw
  lr: 0.0005
  use_amp: True
  use_ema: True
  use_mixup: True
  label_smoothing: 0.1
  weight_decay: 0.05
  grad_clip_norm: 1.0
  accumulation_steps: 1
  device: cuda:0
  gpu: Tesla T4
  gpu_memory_gb: 14.6

Drive 中的文件 (3 个):
  cifar100_best.pth  (73.3 MB)
  cifar100_last.pth  (73.3 MB)
  vit_summary.json  (0.0 MB)


## 8. 断点续训

Colab 断线后重新连接 Runtime，从这里执行即可恢复。

In [20]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, shutil

# 恢复代码
drive_zip = '/content/drive/MyDrive/cifar100_code.zip'
project_dir = '/content/cifar100'
if not os.path.exists(os.path.join(project_dir, 'train.py')):
    os.makedirs(project_dir, exist_ok=True)
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(project_dir)
    print('代码已重新解压')
else:
    print('代码已存在，跳过解压')

# 恢复数据集
dataset_root = '/content/cifar100_data'
drive_dataset = '/content/drive/MyDrive/cifar100_data'
if not os.path.exists(os.path.join(dataset_root, 'cifar-100-python')):
    if os.path.exists(os.path.join(drive_dataset, 'cifar-100-python')):
        print('从 Drive 恢复数据集...')
        shutil.copytree(drive_dataset, dataset_root, dirs_exist_ok=True)
        print('数据集已就绪')
    else:
        print('数据集不存在，请先执行「3. CIFAR-100 数据集」单元')

os.chdir('/content/cifar100')

# 恢复 checkpoint 并续训
resume_path = '/content/drive/MyDrive/cifar100_runs/vit/cifar100_last.pth'
if os.path.exists(resume_path):
    import torch
    ckpt = torch.load(resume_path, map_location='cpu', weights_only=False)
    print(f'找到 checkpoint: epoch {ckpt.get("epoch", "?")}, best_acc {ckpt.get("best_test_accuracy", "?"):.4f}')
    print('开始断点续训...')
    !python train.py --config config_colab_vit.yaml --resume {resume_path} --drive-sync /content/drive/MyDrive/cifar100_runs/vit
else:
    print('未找到 checkpoint，请先执行「6. 启动 ViT 训练」单元')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
代码已存在，跳过解压
找到 checkpoint: epoch 227, best_acc 0.6824
开始断点续训...
GPU: Tesla T4
CUDA 版本: 12.8
PyTorch 版本: 2.11.0+cu128
GPU 显存: 14.6 GB
使用设备: cuda:0
当前模型: vit
模型参数: {'attention_dropout': 0.0, 'depth': 6, 'drop_path_rate': 0.1, 'dropout': 0.1, 'embed_dim': 256, 'image_size': 32, 'in_channels': 3, 'mlp_ratio': 4.0, 'num_classes': 100, 'num_heads': 8, 'patch_size': 4}
AMP 已启用 (torch.amp.GradScaler)
模型参数量: 4,794,212 (4.79M)
可训练参数: 4,794,212 (4.79M)
EMA 已启用, decay=0.999
训练集数量: 50000
测试集数量: 10000
优化器: adamw
从 checkpoint 恢复: /content/drive/MyDrive/cifar100_runs/vit/cifar100_last.pth
恢复到 epoch 227, best_acc=0.6824
模型保存目录: /content/cifar100/models/vit
日志保存目录: /content/cifar100/logs/vit
Mixup alpha=0.2, CutMix alpha=1.0, prob=0.5
Epoch [228/300] | Train Loss: 2.6567 | Train Acc: 0.5043 | Epoch: 00:00:42 | ETA: 00:50:33 | GPU Mem: 5571/6034 MB
         | Test Loss: 1.8054 |